# **Exercise 1: Tokenization with BERT**

In [2]:
# Installing dependecies
# !pip install transformers torch

# Importinng tokenier
from transformers import BertTokenizer

# Loading the tokenizer for the base BERT model
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Choosign a sample sentence
text = "Artifical intelligence is transforming the world"

# Tokenizing and encoding it for BERT
tokens = tokenizer.tokenize(text)
encoded = tokenizer.encode_plus(
    text,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=12,
    return_tensors="pt"
)

# Displayinng results
print("Tokens:", tokens)
print("Token IDs:", encoded["input_ids"])
print("Attenion Mask:", encoded["attention_mask"])

Tokens: ['art', '##ific', '##al', 'intelligence', 'is', 'transforming', 'the', 'world']
Token IDs: tensor([[  101,  2396, 18513,  2389,  4454,  2003, 17903,  1996,  2088,   102,
             0,     0]])
Attenion Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]])


# **Exercise 2: Sentiment Analysis with BERT Pipeline**

In [14]:
# Installing dependecy
! pip install -q transformers torch

# Importing the pipeline helper
from transformers import pipeline

# Creating a sentiment analysis pipeline
clf = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english" # BERT-like model, fine-tuned for sentiment
)

# trying a single sentence
texts = "I absolutely loved the movie - the acting was brilliant!"
results = clf.predict(text)
print(results)

# Trying a small batch
texts = [
    "This product is amazinng and totally worth it."
    "I'm dissappointed. It broke after two days."
    "Meh, it's okay - not great, not terrible"
]

batch_results = clf(texts)
for t, r in zip(texts, batch_results):
  print(f"{t}\n -> {r}\n")

Device set to use cuda:0


[{'label': 'POSITIVE', 'score': 0.999881386756897}]
This product is amazinng and totally worth it.I'm dissappointed. It broke after two days.Meh, it's okay - not great, not terrible
 -> {'label': 'POSITIVE', 'score': 0.9991206526756287}



### Sentiment Analysis for each sentence in a paragraph.

In [13]:
# Step 1: Install dependencies
!pip install transformers torch --quiet

# Step 2: Import pipeline and regex
from transformers import pipeline
import re

# Step 3: Initialize sentiment analysis pipeline
clf = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Step 4: Create a paragraph (single string)
text_block = """
This product is amazing and totally worth it.
I'm disappointed. It broke after two days.
Meh, it's okay - not great, not terrible.
"""

# Step 5: Split the text into sentences
sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text_block) if s.strip()]

# Step 6: Run sentiment analysis on each sentence
results = clf(sentences)

# Step 7: Print results nicely
for s, r in zip(sentences, results):
    print(f"{r['label']:>8} ({r['score']:.3f}) → {s}")

Device set to use cuda:0


POSITIVE (1.000) → This product is amazing and totally worth it.
NEGATIVE (1.000) → I'm disappointed.
NEGATIVE (0.994) → It broke after two days.
POSITIVE (0.994) → Meh, it's okay - not great, not terrible.


# **Exercise 3: Building a Custom Sentiment Analyzer**

In [29]:
# Importing Libraries
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [24]:
# Making a class I can control
class BERTSentimentAnalyzer:
  def __init__(self, model_name="distilbert-base-uncased-finetuned-sst-2-english"):
    # Loading tokenizer + model weights
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
    # Move model to GPU if available
    self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.model.to(self.device)
    # Label mapping (SST-2 = 0: NEGATIVE, 1: POSITIVE)
    self.id2label = self.model.config.id2label
    self.label2id = self.model.config.label2id

  def preprocess(self, texts, max_len=256):
    """
    Clean → tokenize → build tensors.
    Accepts a string or list of strings.
    Returns a dict of tensors on the correct device.
    """
    if isinstance(texts, str):
      texts = [texts]

    enc = self.tokenizer(
        texts,
        padding=True,              # Pad to longest in the batch
        truncation=True,          # cut long sequences
        max_length=max_len,
        return_tensors="pt"       # get PyTorch tensors directly
  )

    # Moving to GPU/CPU
    return {k: v.to(self.device) for k, v in enc.items()}

  def predict(self, texts, max_len=256):
      """
      Forward pass → logits → softmax → labels + scores.
      """
      batch = self.preprocess(texts, max_len=max_len)

      self.model.eval()
      with torch.no_grad():
        outputs = self.model(**batch)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)
        conf, pred_ids = torch.max(probs, dim=-1)

        # Convertiong ids - labels and out on CPU as python types
        labels = [self.id2label[int(i)] for i in pred_ids]
        scores = [float(c) for c in conf]
        return [{"label": lab, "score": sc} for lab, sc in zip(labels, scores)]

In [25]:
# Trying it out

clf = BERTSentimentAnalyzer()

# single text
print(clf.predict("I absolutely loved the app—super smooth experience!"))
# batch
texts = [
    "Worst customer service ever.",
    "It works, but it's a bit slow.",
    "What a fantastic update. Thank you!"
]
print(clf.predict(texts))

[{'label': 'POSITIVE', 'score': 0.9998301267623901}]
[{'label': 'NEGATIVE', 'score': 0.999788224697113}, {'label': 'NEGATIVE', 'score': 0.9952991008758545}, {'label': 'POSITIVE', 'score': 0.999876856803894}]


# **Exercise 4: Understanding BERT for Named Entity Recognition (NER)**

In [31]:
!pip install -q transformers torch

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
import numpy as np

In [32]:
class BERTNamedEntityRecognizer:
    def __init__(self, model_name="dslim/bert-base-NER"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model     = AutoModelForTokenClassification.from_pretrained(model_name)
        self.id2label  = self.model.config.id2label
        self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def _postprocess(self, text, tokens, labels):
        """
        Merge WordPiece sub-tokens back into whole words and group BIO tags
        into entity spans with {text, label, start, end}.
        """
        entities = []
        current = None

        # tokenizer gives offsets when we request them
        for tok, lab in zip(tokens, labels):
            tag = lab
            if tag == "O":
                # close running entity if any
                if current:
                    entities.append(current)
                    current = None
                continue

            # tag looks like "B-ORG" or "I-PER"
            prefix, ent_type = tag.split("-", 1)

            if prefix == "B":
                # start a new entity
                if current:
                    entities.append(current)
                current = {"label": ent_type, "text": tok["text"], "start": tok["start"], "end": tok["end"]}
            elif prefix == "I" and current and current["label"] == ent_type:
                # continue same entity
                current["text"] += text[current["end"]:tok["start"]] + tok["text"]  # keep spaces
                current["end"]   = tok["end"]
            else:
                # inconsistent I- tag (start a new one)
                if current:
                    entities.append(current)
                current = {"label": ent_type, "text": tok["text"], "start": tok["start"], "end": tok["end"]}

        if current:
            entities.append(current)
        return entities

    def predict(self, text, aggregation=True):
        """
        Returns either per-token tags or merged entities.
        """
        enc = self.tokenizer(
            text,
            return_offsets_mapping=True,   # we need character spans
            return_tensors="pt",
            truncation=True
        )
        offset_mapping = enc.pop("offset_mapping")[0].tolist()

        enc = {k: v.to(self.device) for k, v in enc.items()}
        self.model.eval()
        with torch.no_grad():
            out    = self.model(**enc)
            logits = out.logits[0].cpu().numpy()     # [seq_len, num_labels]
            preds  = np.argmax(logits, axis=-1)      # label ids

        # Map ids → labels and keep only real tokens (skip special tokens)
        input_ids = enc["input_ids"][0].cpu().tolist()
        tokens    = self.tokenizer.convert_ids_to_tokens(input_ids)

        visible_tokens = []
        visible_labels = []
        for tok, off, lab_id in zip(tokens, offset_mapping, preds):
            start, end = off
            # skip special tokens like [CLS], [SEP] (offset 0,0)
            if end == 0 and start == 0:
                continue
            # skip pure continuation tokens when they have zero-length (shouldn’t happen with offsets)
            piece_text = text[start:end]
            visible_tokens.append({"text": piece_text, "start": start, "end": end})
            visible_labels.append(self.id2label[int(lab_id)])

        if not aggregation:
            # Return raw per-token predictions
            return [{"text": t["text"], "label": l, "start": t["start"], "end": t["end"]}
                    for t, l in zip(visible_tokens, visible_labels)]

        # Merge subwords using BIO logic
        return self._postprocess(text, visible_tokens, visible_labels)

In [33]:
ner = BERTNamedEntityRecognizer()

text = "Barack Obama met Sundar Pichai at Google HQ in Mountain View on January 5, 2020."
entities = ner.predict(text, aggregation=True)
entities

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'label': 'PER', 'text': 'Barack Obama', 'start': 0, 'end': 12},
 {'label': 'PER', 'text': 'Sun', 'start': 17, 'end': 20},
 {'label': 'PER', 'text': 'dar Pichai', 'start': 20, 'end': 30},
 {'label': 'ORG', 'text': 'Google', 'start': 34, 'end': 40},
 {'label': 'LOC', 'text': 'Mountain View', 'start': 47, 'end': 60}]

# **Exercise 5: Comparing BERT and GPT**

| Dimension | BERT | GPT |
|---|---|---|
| Core architecture | Encoder (bidirectional self-attention) | Decoder-only (causal, left-to-right attention) |
| Directionality | Bidirectional (sees left & right context) | Unidirectional (predicts next token from left context) |
| Pretraining task | Masked Language Modeling (MLM) (+ NSP in original) | Causal Language Modeling (CLM) — next-token prediction |
| Primary purpose | Understanding (representations) | Generation (text production) |
| Typical tasks | NER, sentiment, classification, QA (extractive), retrieval embeddings | Chat, summarization, drafting, code gen, QA (generative), agents |
| Input handling | Strong for fixed inputs you want to label/extract | Strong for long-form outputs & multi-turn dialogue |
| Outputs | Usually labels or embeddings | Fluent text (token stream) |
| Fine-tuning style | Task-specific heads on encoder; often lightweight | Instruction tuning, RLHF, adapters, LoRA, SFT |
| Strengths | Precise token-level understanding; robust embeddings; efficient for classification | Flexible generation; reasoning chains; tool use; few-shot prompting |
| Weaknesses | Not a native generator; original BERT has shorter context; NSP is debated | Can hallucinate; sensitive to prompts; compute-heavy for training |
| Best when… | You need high-quality features/embeddings or token-level decisions | You need coherent outputs, step-by-step reasoning, or open-ended tasks |

# **Exercise 6: Exploring BERT Applications in Retrieval-Augmented Generation (RAG)**

# What Does BERT Do in RAG?

**BERT** acts as the **retriever’s brain** — it’s used to convert text into **embeddings** (numerical representations that capture meaning).

---

## Example: How Retrieval Works

**User query:**  
> “What are the side effects of ibuprofen?”

### Step-by-step process:
1. The query (“side effects of ibuprofen”) is converted into an **embedding vector** using **BERT**.  
2. All documents in your **knowledge base** are also converted into embeddings using the same BERT model.  
3. A **vector database** (like **FAISS**, **Pinecone**, or **Chroma**) compares vectors and finds documents whose embeddings are *closest* to the query.  
4. Those top matches are sent to a **generator model** (like GPT or LLaMA) to produce the final, human-readable answer.

---

## Summary of Roles

| Component | Role |
|------------|------|
|  **BERT** | Embeds queries and documents into numerical vectors (Retriever) |
|  **Vector DB** | Stores and searches embeddings to find similar content |
|  **GPT / LLaMA** | Generates the final answer based on the retrieved context |

---

##  How BERT Generates Embeddings

BERT takes text and transforms it into **high-dimensional vectors** that capture **semantic meaning** (context and relationships).

**Example:**

| Word | Embedding (simplified) |
|------|------------------------|
| "doctor" | `[0.12, 0.45, -0.33, ...]` |
| "hospital" | `[0.10, 0.44, -0.35, ...]` |

Because these vectors are **close together**, BERT “understands” that **doctor** and **hospital** are semantically related.  
That’s what makes it powerful for **retrieval tasks** — it captures *meaning*, not just *keywords*.

---

##  How the Vector Database Works

The **vector database** stores all document embeddings.  
When a query comes in:

1. It embeds the query using BERT.  
2. It calculates **cosine similarity** between the query vector and each stored document vector.  
3. It returns the **most similar documents** — the ones closest in semantic meaning.

>  *Cosine similarity* measures how close two meanings are — a smaller “distance” = a stronger semantic relationship.

---

###  Recap Diagram


| Step | Model            | Purpose                                                  |
|-----:|------------------|----------------------------------------------------------|
|  1 | **BERT (Encoder)** | Converts queries + documents into embeddings for retrieval. |
|  2 | **Vector DB**       | Finds semantically relevant documents.                    |
|  3 | **GPT (Decoder)**   | Reads retrieved text and generates a final, human-readable answer. |